# Single-organoid graph build + crypt segmentation (debug notebook)

This notebook runs the **exact same core pipeline** as the dataset scripts, but for **one organoid** so you can quickly iterate on segmentation hyperparameters.

- Stage 1: build/load a cell graph (nodes=cells, edges=adjacency) and attach HKS + vocab encoding.
- Stage 2: run crypt segmentation using vocab indices.

Notes:
- Variables are intentionally kept **exposed** (minimal functions) so you can insert cells to inspect intermediate arrays.
- Paths below mirror the defaults in the provided scripts.


In [ ]:
# --- Imports ---
import os, glob
import numpy as np
import pandas as pd

# organograph core objects
from organograph.mesh.OrganoidMesh import OrganoidMesh
from organograph.mesh.transform import ensure_mesh_graph_aligned

# graph build + IO
from organograph.graph.build import build_organoid_graph, add_vertex_field_to_graph
from organograph.graph.io import load_cell_graph, save_cell_graph

# cells table / nuclei extraction
from organograph.io_utils.cells_table import prepare_cells_table, make_nuclei_extractor, suppress_marker_if_coexpressed

# geometry features / vocab
from organograph.mesh.hks import compute_hks
from organograph.crypts.vocab import compute_vocabulary_encoding

# crypt segmentation
from organograph.crypts.segment import segment_crypts_organoid

# segmentation IO (optional save)
from organograph.io_utils.segmentation_io import save_segmentation_npz, patches_to_ll


## 1) Paths + dataset config
These defaults follow the two scripts you attached. Adjust as needed.


In [ ]:
# --- Locate project root relative to the scripts you provided (recommended) ---
# If you run this notebook from the project repo, the following should work:
SCRIPT_GRAPH_PREPROCESS = os.path.abspath("run_graph_preprocess.py")   # adjust if needed
SCRIPT_SEGMENTATION     = os.path.abspath("run_crypt_segmentation.py") # adjust if needed

# Project root = parent of scripts folder, per your scripts.
# If you keep scripts in <repo>/scripts/, then PROJECT_ROOT=<repo>.
# If you're running elsewhere, set PROJECT_ROOT manually.
_SCRIPT_DIR = os.path.dirname(SCRIPT_GRAPH_PREPROCESS)
PROJECT_ROOT = os.path.dirname(_SCRIPT_DIR)

# --- Data locations (match defaults in scripts) ---
MESH_DATA_DIR  = os.path.join(PROJECT_ROOT, "..", "NicoleData", "20251201", "fractal_output")
CELLS_CSV      = os.path.join(PROJECT_ROOT, "..", "NicoleData", "20251201", "cell_features_class.csv")
VOCAB_PATH     = os.path.join(PROJECT_ROOT, "sim", "vocab.npz")

# Where to read/write graphs
OUT_GRAPHS_DIR = os.path.join(PROJECT_ROOT, "..", "NicoleData", "20251201", "graphs_preprocessed")

# Where to save segmentation results (optional)
SEG_DIR        = os.path.join(PROJECT_ROOT, "..", "NicoleData", "20251201", "crypt_segmentations")

# --- Timepoint folder you want to work with ---
TIMEPOINT = "day4p5"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_GRAPHS_DIR:", OUT_GRAPHS_DIR)
print("SEG_DIR:", SEG_DIR)


## 2) Choose an organoid
Two common ways:
1) **Load an existing graph** from `graphs_preprocessed/<timepoint>/<label_uid>.gpickle`.
2) **Build a graph from a mesh + the nuclei table** (then optionally save it).

If you already ran preprocessing, option (1) is fastest.


In [ ]:
# --- Option (1): load an existing precomputed graph ---
# Provide a label_uid and load the graph.
label_uid = None  # e.g. "B02_f01_000123" (set this)

# If label_uid is None, we list a few candidates from the folder to help you pick.
tp_dir = os.path.join(OUT_GRAPHS_DIR, TIMEPOINT)
gpickles = sorted(glob.glob(os.path.join(tp_dir, "*.gpickle")))

print(f"Found {len(gpickles)} graphs under: {tp_dir}")
print("First 10:")
for p in gpickles[:10]:
    print("  ", os.path.basename(p))

# If you want to auto-pick the first one (useful for smoke tests):
if label_uid is None and len(gpickles) > 0:
    label_uid = os.path.splitext(os.path.basename(gpickles[0]))[0]
    print("\nAuto-picked label_uid =", label_uid)

graph_path = os.path.join(tp_dir, f"{label_uid}.gpickle")
print("graph_path =", graph_path)


In [ ]:
# Load the graph
G = load_cell_graph(graph_path)

print("Loaded graph with:")
print("  nodes:", G.number_of_nodes())
print("  edges:", G.number_of_edges())
print("  graph keys:", list(G.graph.keys())[:20])

# Helpful: inspect one node's attributes (you can change node_id)
node_id = next(iter(G.nodes))
print("\nExample node:", node_id)
print("Node keys:", list(G.nodes[node_id].keys()))


## 3) Resolve + load the mesh for the same organoid
The segmentation stage needs the mesh. Ideally `G.graph['mesh_path']` is set (as in your dataset segmentation script).


In [ ]:
# Resolve mesh path from the graph metadata
mesh_path = G.graph.get("mesh_path", None)
print("mesh_path in graph:", mesh_path)

# If mesh_path is missing, you can set it manually:
# mesh_path = "path/to/organoid.vtp"

assert mesh_path is not None, "mesh_path missing. Either rebuild graphs with mesh_path stored, or set it manually here."
assert os.path.exists(mesh_path), f"mesh file not found: {mesh_path}"

mesh = OrganoidMesh(mesh_path)

# In preprocessing you normalize the mesh before building the graph.
# Here we only need consistent alignment between mesh and graph.
# If your mesh on disk is not normalized already, you may need:
# mesh.normalize_inplace()

print("Mesh loaded.")
print("  n_vertices:", getattr(mesh, "n_vertices", None) or mesh.V.shape[0] if hasattr(mesh, "V") else "unknown")


In [ ]:
# Ensure mesh/graph are aligned (same coordinate conventions etc.)
ensure_mesh_graph_aligned(mesh, G)
print("Mesh/graph alignment check passed.")


## 4) (Optional) Build the graph from scratch instead of loading
If you want to test the full Stage-1 preprocessing for a single organoid, run the next section.
This will:
- load the cells table,
- construct nuclei extractor,
- build the adjacency graph from the mesh,
- attach HKS and vocab encoding.

If you loaded `G` above and it already has these fields, you can skip.


In [ ]:
# --- Marker configuration (matches run_graph_preprocess.py defaults) ---
COORD_COLS = ("0.x_pos_pix", "0.y_pos_pix", "0.z_pos_pix_scaled")
MARKER_COLS = [
    "0.C02.percentile99_class",  # LGR5
    "0.C03.percentile99_class",  # chroma
    "0.C04.percentile99_class",  # aldoB
    "1.C02.percentile99_class",  # Sero
    "1.C03.percentile99_class",  # Lyz
    "1.C04.percentile99_class",  # Agr2
    "2.C04.percentile99_class",  # ki67
]
LGR5_MARKER = "0.C02.percentile99_class"
COEXP_MARKERS = (
    "1.C03.percentile99_class",  # Lyz
    "1.C04.percentile99_class",  # Agr2
    "1.C02.percentile99_class",  # Sero
    "0.C03.percentile99_class",  # chroma
)

HKS_TIMES = [1.0, 2.0, 4.0, 8.0, 25.0]

def marker_postprocess(markers_bin, marker_names):
    return suppress_marker_if_coexpressed(
        markers_bin,
        marker_names,
        exclusive_marker=LGR5_MARKER,
        forbidden_markers=COEXP_MARKERS,
        copy=True,
        ignore_missing=True,
    )

# --- Load cells table once ---
cells_df = pd.read_csv(CELLS_CSV)
cells_df = prepare_cells_table(cells_df, label_col="label_uid")

# --- Extractor: given label_uid -> (xyz_raw, markers_bin, marker_names) ---
extractor = make_nuclei_extractor(
    cells_df,
    label_col="label_uid",
    xyz_cols=COORD_COLS,
    marker_cols=MARKER_COLS,
    marker_postprocess_fn=marker_postprocess,
)

# --- (Re)load mesh and normalize (as preprocessing does) ---
mesh2 = OrganoidMesh(mesh_path)
mesh2.normalize_inplace()
mesh2.label_uid = label_uid

# --- Build graph ---
G2, aux = build_organoid_graph(mesh=mesh2, extract_fn=extractor)

print("Built graph:")
print("  nodes:", G2.number_of_nodes())
print("  edges:", G2.number_of_edges())
print("Aux keys:", list(aux.keys())[:20])


In [ ]:
# --- Attach HKS (required downstream) ---
mesh2._eig_decomp()  # Laplace eigenpairs
hks = compute_hks(mesh2, t=HKS_TIMES, coeffs=False)  # (V, T)
add_vertex_field_to_graph(G2, hks, "hks")
G2.graph["hks_times"] = np.asarray(HKS_TIMES, float)

# --- Attach vocab encoding ---
vocab = np.load(VOCAB_PATH, allow_pickle=True)
encoding, _, _, _ = compute_vocabulary_encoding(vocab, mesh2)
add_vertex_field_to_graph(G2, encoding, "vocab_encoding")
G2.graph["vocab_path"] = VOCAB_PATH

# Record mesh path in graph metadata (this is what segmentation script expects)
G2.graph["mesh_path"] = mesh_path
G2.graph["label_uid"] = label_uid

print("Attached fields: hks + vocab_encoding")
print("Example node keys:", list(G2.nodes[next(iter(G2.nodes))].keys()))


In [ ]:
# --- Choose which graph object to use going forward ---
# If you rebuilt the graph above, switch to G2; otherwise keep the loaded G.
USE_REBUILT_GRAPH = False

if USE_REBUILT_GRAPH:
    G = G2
    mesh = mesh2
    print("Using rebuilt graph G2 + mesh2.")
else:
    print("Using loaded graph G + mesh.")


## 5) Crypt segmentation (single organoid)
This is Stage 2 from `run_crypt_segmentation.py`, but run in-notebook so you can tweak knobs.


In [ ]:
# --- Canonical axis / binning (matches run_crypt_segmentation.py defaults) ---
S = 1.5
n_bins = 80
bin_edges = np.linspace(0.0, S, n_bins + 1)
BIN_CENTERS = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_centers = np.asarray(BIN_CENTERS, float)

# --- Vocab indices used for seeding ---
CRYPT_VOCAB_IDX = [1, 2, 7]
NECK_VOCAB_IDX  = [0, 3, 4]  # or None

# --- Hyperparameters to tune ---
# These are forwarded to segment_crypts_organoid(...) as kwargs.
SEGMENT_KWARGS = dict(
    # crypt_seed_thresh=0.2,
    # neck_seed_thresh=0.5,
    # min_crypt_seed_size=10,
    # grow_steps=2,
)

DEBUG = False  # set True to get additional debug output from the segmentation code (if supported)

print("BIN_CENTERS shape:", bin_centers.shape)
print("CRYPT_VOCAB_IDX:", CRYPT_VOCAB_IDX)
print("NECK_VOCAB_IDX :", NECK_VOCAB_IDX)
print("SEGMENT_KWARGS :", SEGMENT_KWARGS)


In [ ]:
# --- Run segmentation ---
out = segment_crypts_organoid(
    G=G,
    mesh=mesh,
    bin_centers=bin_centers,
    crypt_vocab_idx=list(CRYPT_VOCAB_IDX),
    neck_vocab_idx=None if NECK_VOCAB_IDX is None else list(NECK_VOCAB_IDX),
    debug=DEBUG,
    **(SEGMENT_KWARGS or {}),
)

if DEBUG:
    crypts, villi, d_norm, L_crypt, Circ, dbg = out
else:
    crypts, villi, d_norm, L_crypt, Circ = out
    dbg = None

print("Segmentation done.")
print("  n_crypts:", len(crypts))
print("  n_villi :", len(villi))
print("  d_norm shape:", np.asarray(d_norm).shape)
print("  L_crypt shape:", np.asarray(L_crypt).shape)
print("  Circ shape:", np.asarray(Circ).shape)


## 6) Save segmentation output (optional)
This matches the dataset pipeline output format so you can reuse existing downstream plotting/analysis.


In [ ]:
# --- Save ---
SAVE = False  # flip to True when you want outputs saved

if SAVE:
    os.makedirs(os.path.join(SEG_DIR, TIMEPOINT), exist_ok=True)
    seg_path = os.path.join(SEG_DIR, TIMEPOINT, f"{label_uid}.npz")

    save_segmentation_npz(
        seg_path,
        label_uid=label_uid,
        crypts_ll=patches_to_ll(crypts),
        villi_ll=patches_to_ll(villi),
        bin_centers=bin_centers,
        d_norm=d_norm,
        L_crypt=L_crypt,
        Circ=Circ,
        extra={"debug": dbg, "graph_path": graph_path, "mesh_path": mesh_path} if dbg is not None else
              {"graph_path": graph_path, "mesh_path": mesh_path},
    )
    print("Saved:", seg_path)
else:
    print("Skipping save (SAVE=False).")


## 7) Next: plotting
You said you'll add plotting afterwards. Suggested quick checks to add later:
- visualize crypt/villus patches on the mesh
- inspect seed scores, thresholds, and growth iterations
- sanity check vocab_encoding distributions for CRYPT_VOCAB_IDX
